<a href="https://colab.research.google.com/github/Architag1503/Colab/blob/main/FPGrowthAlgorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from collections import defaultdict

# -----------------------------
# FP-Tree Node Definition
# -----------------------------
class FPNode:
    def __init__(self, item, count, parent):
        self.item = item
        self.count = count
        self.parent = parent
        self.children = {}
        self.link = None   # link to next node with same item

In [2]:
# -----------------------------
# Update Header Table (Linking nodes)
# -----------------------------
def update_header(header, item, node):
    if header[item] is None:
        header[item] = node
    else:
        current = header[item]
        while current.link:
            current = current.link
        current.link = node

In [3]:
# -----------------------------
# Insert transaction into FP-Tree
# -----------------------------
def insert_tree(items, node, header):
    if len(items) == 0:
        return

    first = items[0]

    # If child exists → increase count
    if first in node.children:
        node.children[first].count += 1
    else:
        # Create new node
        new_node = FPNode(first, 1, node)
        node.children[first] = new_node

        # Update header table
        update_header(header, first, new_node)

    # Recurse for remaining items
    insert_tree(items[1:], node.children[first], header)


In [4]:
# -----------------------------
# Build FP-Tree
# -----------------------------
def build_fp_tree(transactions, min_support):
    # Step 1: Count frequency
    freq = defaultdict(int)
    for transaction in transactions:
        for item in transaction:
            freq[item] += 1

    # Step 2: Remove infrequent items
    freq = {item: count for item, count in freq.items() if count >= min_support}

    if len(freq) == 0:
        return None, None

    # Step 3: Header table
    header = {item: None for item in freq}

    # Step 4: Create root
    root = FPNode(None, 1, None)

    # Step 5: Insert transactions
    for transaction in transactions:
        # Filter + sort by frequency (descending)
        ordered = [item for item in transaction if item in freq]
        ordered.sort(key=lambda x: freq[x], reverse=True)

        insert_tree(ordered, root, header)

    return root, header



In [5]:
# -----------------------------
# Get Conditional Pattern Base
# -----------------------------
def get_conditional_pattern_base(item, header):
    patterns = []
    node = header[item]

    while node:
        path = []
        parent = node.parent

        # Traverse up to root
        while parent and parent.item is not None:
            path.append(parent.item)
            parent = parent.parent

        # Add path count times
        for _ in range(node.count):
            patterns.append(path)

        node = node.link

    return patterns


In [6]:
# -----------------------------
# Mine FP-Tree recursively
# -----------------------------
def mine_fp_tree(header, min_support, prefix, frequent_patterns):
    # Sort items (least frequent first)
    items = sorted(header.keys(), key=lambda x: x)

    for item in items:
        new_pattern = prefix.copy()
        new_pattern.add(item)

        # Save pattern
        frequent_patterns.append(new_pattern)

        # Get conditional pattern base
        conditional_patterns = get_conditional_pattern_base(item, header)

        # Build conditional FP-tree
        cond_tree, cond_header = build_fp_tree(conditional_patterns, min_support)

        # Recursive mining
        if cond_header:
            mine_fp_tree(cond_header, min_support, new_pattern, frequent_patterns)



In [16]:
def print_tree(node, indent=0):
    if node is None:
        return

    # Print current node
    print("  " * indent + f"{node.item}:{node.count}")

    # Recursively print children
    for child in node.children.values():
        print_tree(child, indent + 1)

In [17]:
def print_header_table(header):
    print("\nHeader Table Chains:\n")

    for item, node in header.items():
        print(f"{item} → ", end="")

        current = node
        while current:
            print(f"[count={current.count}]", end=" -> ")
            current = current.link

        print("None")

In [7]:

# -----------------------------
# Main Function
# -----------------------------
def fp_growth(transactions, min_support):
    tree, header = build_fp_tree(transactions, min_support)

    frequent_patterns = []
    if header:
        mine_fp_tree(header, min_support, set(), frequent_patterns)

    return frequent_patterns

In [19]:
# -----------------------------
# Example Usage
# -----------------------------
if __name__ == "__main__":
    transactions = [
        ['Bread', 'Milk', 'Rusk'],
        ['Bread', 'Butter', 'Jam'],
        ['Milk', 'Butter'],
        ['Bread', 'Milk', 'Butter'],
        ['Bread', 'Milk', 'Rusk'],
        ['Bread', 'Milk', 'Jam'],
        ['Bread', 'Butter', 'Jam', 'Rusk'],
        ['Milk', 'Butter', 'Rusk'],
        ['Bread', 'Milk', 'Butter'],
        ['Bread', 'Milk'],
        ['Bread', 'Butter', 'Jam'],
        ['Milk', 'Butter', 'Bread', 'Rusk'],
        ['Bread', 'Milk', 'Rusk'],
        ['Bread', 'Butter', 'Jam'],
        ['Milk', 'Butter', 'Bread', 'Rusk'],
    ]

    min_support = 2

    patterns = fp_growth(transactions, min_support)

    print("Frequent Patterns:")
    for p in patterns:
        print(p)

    tree, header = build_fp_tree(transactions, min_support)

    print("FP-Tree Structure:\n")
    print_tree(tree)

    print_header_table(header)

Frequent Patterns:
{'Bread'}
{'Butter'}
{'Bread', 'Butter'}
{'Butter', 'Milk'}
{'Bread', 'Butter', 'Milk'}
{'Jam'}
{'Jam', 'Bread'}
{'Jam', 'Butter'}
{'Jam', 'Butter', 'Bread'}
{'Milk'}
{'Bread', 'Milk'}
{'Rusk'}
{'Rusk', 'Bread'}
{'Rusk', 'Bread', 'Milk'}
{'Rusk', 'Butter'}
{'Rusk', 'Bread', 'Butter'}
{'Rusk', 'Butter', 'Milk'}
{'Rusk', 'Bread', 'Butter', 'Milk'}
{'Rusk', 'Milk'}
FP-Tree Structure:

None:1
  Bread:13
    Milk:9
      Rusk:3
      Butter:4
        Rusk:2
      Jam:1
    Butter:4
      Jam:3
      Rusk:1
        Jam:1
  Milk:2
    Butter:2
      Rusk:1

Header Table Chains:

Bread → [count=13] -> None
Milk → [count=9] -> [count=2] -> None
Rusk → [count=3] -> [count=1] -> [count=1] -> [count=2] -> None
Butter → [count=4] -> [count=2] -> [count=4] -> None
Jam → [count=3] -> [count=1] -> [count=1] -> None
